# FastML Tutorial – Jet Tagging with hls4ml

Identify jets from the LHC using a neural network  
and deploy the model to FPGA firmware using hls4ml.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nairods/Scies4Free-FastML-tutorial/blob/main/FastML.ipynb)

# 1. Setup

We begin by importing the required Python libraries for:
- Dataset handling
- Preprocessing
- Neural network training

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.utils import to_categorical
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

%matplotlib inline
seed = 42

## 1.2 Load the Jet Tagging Dataset

We use the **hls4ml LHC jet dataset** from OpenML.

The dataset contains:
- 830,000 jet events
- 16 high-level physics features
- 5 jet classes:
  - quark (q)
  - gluon (g)
  - W boson (w)
  - Z boson (z)
  - top quark (t)

In [ ]:
data = fetch_openml('hls4ml_lhc_jets_hlf')
X, y = data['data'], data['target']

### Let's print some information about the dataset
Print the feature names and the dataset shape

In [ ]:
print(data['feature_names'])
print(X.shape, y.shape)
print(X[:5])
print(y[:5])

## 1.3 Encode Labels and Split Dataset

As you saw above, the `y` target is an array of strings, e.g. \['g', 'w',...\] etc.  
We map them to integers and split the dataset into training and validation sets:

- 70% training
- 15% validation
- 15% test

with equal representation of classes by using stratify.

In [ ]:
# Convert y strings into integers
codecs = {'g': 0, 'q': 1, 't': 4, 'w': 2, 'z': 3}
y = np.array([codecs[i] for i in y])

# Split data in train vs val_test
X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)

# Split val_test in half → val and test
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5, random_state=seed, stratify=y_val_test)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

## 1.4 Feature Scaling

Neural networks train better when input features are standardized.

We:
- Fit the scaler on training data only
- Apply the same transformation to validation and test sets

In [ ]:
# Convert to float32
X_train = X_train.astype(np.float32)
X_val   = X_val.astype(np.float32)
X_test  = X_test.astype(np.float32)

# Scale using training data only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

In case the automatic download does not work, a local backup of the dataset that can be loaded instead:

In [ ]:
# !git clone https://github.com/nairods/Scies4Free-FastML-tutorial.git
# DATA_PATH = "Scies4Free-FastML-tutorial/data/"

# train = np.load(DATA_PATH + "train.npz", allow_pickle=True)
# val   = np.load(DATA_PATH + "val.npz", allow_pickle=True)
# test  = np.load(DATA_PATH + "test.npz", allow_pickle=True)

# feature_names = ['zlogz', 'c1_b0_mmdt', 'c1_b1_mmdt', 'c1_b2_mmdt', 'c2_b1_mmdt', 'c2_b2_mmdt', 'd2_b1_mmdt', 'd2_b2_mmdt', 'd2_a1_b1_mmdt', 'd2_a1_b2_mmdt', 'm2_b1_mmdt', 'm2_b2_mmdt', 'n2_b1_mmdt', 'n2_b2_mmdt', 'mass_mmdt', 'multiplicity']
# print(feature_names)

# X_train, y_train = train["X"], train["y"]
# X_val,   y_val   = val["X"],   val["y"]
# X_test,  y_test  = test["X"],  test["y"]

We need to convert the class labels into one-hot vectors so that they match the softmax output of the network  
and can be properly compared using categorical cross-entropy loss.

One-hot encoding represents each class as a vector with a 1 at the correct class position and 0 elsewhere: 
- class 0 -> (1,0,0,0,0)
- class 1 -> (0,1,0,0,0)
- class 2 -> (0,0,1,0,0)
- class 3 -> (0,0,0,1,0)
- class 4 -> (0,0,0,0,1)

In [ ]:
# Convert Labels to One-Hot representation
y_train = to_categorical(y_train, 5) 
y_val = to_categorical(y_val, 5) 
y_test = to_categorical(y_test, 5)

# 2. Build the Neural Network

Architecture:
- 3 hidden layers with 64, then 32, then 32 neurons
- Each with a ReLU activation
- 5 output neurons (one for each class)
- Finish with a Softmax activation

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1
from callbacks import all_callbacks

In [ ]:
model = Sequential()
model.add(Dense(64, input_shape=(16,), name='fc1', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu1'))
model.add(Dense(32, name='fc2', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu2'))
model.add(Dense(32, name='fc3', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='relu', name='relu3'))
model.add(Dense(5, name='output', kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001)))
model.add(Activation(activation='softmax', name='softmax'))

## 2.1 Train the model
We train the neural network using the Adam optimizer and categorical cross-entropy loss.

During training:
- The learning rate is automatically reduced when the validation performance plateaus.
- The best-performing model is saved to the directory `model_1`.

The network architecture is relatively small, so training should complete within a few minutes, even on a CPU.

If you have already trained the model and restarted the notebook kernel, you can set `train = False` to skip training and load the previously saved model instead.

In [ ]:
train = True
if train:
    adam = Adam(lr=0.0001)
    model.compile(optimizer=adam, loss=['categorical_crossentropy'], metrics=['accuracy'])
    callbacks = all_callbacks(
        stop_patience=1000,
        lr_factor=0.5,
        lr_patience=10,
        lr_epsilon=0.000001,
        lr_cooldown=2,
        lr_minimum=0.0000001,
        outputDir='model_1',
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=1024,
        epochs=10,
        shuffle=True
    )
else:
    from tensorflow.keras.models import load_model

    model = load_model('model_1/KERAS_check_best_model.h5')

## 2.2 Check performance
Check the accuracy and make a Receiver Operating Characteristic (ROC) curve

In [ ]:
import plotting
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

y_keras = model.predict(X_test)
print("Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
plt.figure(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)

# 3. Convert to FPGA Firmware with hls4ml

Next, we will convert the trained Keras model into a **low-latency FPGA implementation** using `hls4ml`.  
First, we will check that the classification performance remains accurate when using **fixed-point data types**.  
Then, we will synthesize the model with **Vitis HLS** and inspect its **latency and FPGA resource usage**.

### Create an hls4ml configuration and model

The `hls4ml` library uses a **configuration dictionary** to control how the neural network is translated to FPGA firmware.  
Here, we will show a **simple configuration**, however more advanced settings are possible.

In [ ]:
import hls4ml

config = hls4ml.utils.config_from_keras_model(model, granularity='model', backend='Vitis')
print("-----------------------------------")
print("Configuration")
plotting.print_dict(config)
print("-----------------------------------")
hls_model = hls4ml.converters.convert_from_keras_model(
    model, hls_config=config, backend='Vitis', output_dir='model_1/hls4ml_prj', part='xcu250-figd2104-2L-e'
)

Let's visualise what we created. The model architecture is shown, annotated with the shape and data types

In [ ]:
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

## 3.1 Compile, predict
Now we need to check that this model performance is still good. We compile the hls_model, and then use `hls_model.predict` to execute the FPGA firmware with bit-accurate emulation on the CPU.

In [ ]:
hls_model.compile()
X_test = np.ascontiguousarray(X_test)
y_hls = hls_model.predict(X_test)

## 3.2 Compare
That was easy! Now let's see how the performance compares to Keras:

In [ ]:
print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))

fig, ax = plt.subplots(figsize=(9, 9))
_ = plotting.makeRoc(y_test, y_keras, le.classes_)
plt.gca().set_prop_cycle(None)  # reset the colors
_ = plotting.makeRoc(y_test, y_hls, le.classes_, linestyle='--')

from matplotlib.lines import Line2D

lines = [Line2D([0], [0], ls='-'), Line2D([0], [0], ls='--')]
from matplotlib.legend import Legend

leg = Legend(ax, lines, labels=['keras', 'hls4ml'], loc='lower right', frameon=False)
ax.add_artist(leg)

# 4. Synthesize
Now we will use **Vitis HLS** to synthesize the model into FPGA firmware.  
We can start the build directly from our `hls_model` object.  

After synthesis, the generated IP can be integrated into a workflow to compile for a specific FPGA board.  
For this tutorial, we will focus on **reviewing the reports** generated by Vitis HLS, paying attention to **latency** and **resource usage**.

**Note:** This step can take several minutes.

In [ ]:
hls_model.build(csim=False)

## 4.1 Check the reports
Print out the reports generated by Vitis HLS. Pay attention to the Latency and the 'Utilization Estimates' sections

In [ ]:
hls4ml.report.read_vivado_report('model_1/hls4ml_prj/')